# Part 2: Feature Engineering & Data Preparation

The ZomaThon Cart Super Add-On (CSAO) prediction task requires us to featurize the context of an ongoing cart. This notebook demonstrates how raw transaction logs are transformed into dense feature vectors suitable for our LightGBM ranker.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 50)

## 1. Load Raw Entities
We start by loading users, restaurants, and the historical orders.

In [ ]:
users_df = pd.read_parquet("../data/raw/users_adv.parquet")
restaurants_df = pd.read_parquet("../data/raw/restaurants_adv.parquet")
items_df = pd.read_parquet("../data/raw/items_adv.parquet")
orders_df = pd.read_parquet("../data/raw/orders_exploded_adv.parquet")

## 2. Generating User Features
We need to know the historical context of the user. Are they a vegetarian? What is their typical basket size? Are they price sensitive?

In [ ]:
def extract_user_features(orders, items):
    # Merge orders with item info to determine context
    df = orders.merge(items[['item_id', 'price', 'is_veg']], on='item_id')
    
    user_stats = df.groupby('user_id').agg(
        total_orders=('order_id', 'nunique'),
        avg_order_value=('price', 'sum'),
        veg_item_ratio=('is_veg', 'mean')
    ).reset_index()
    
    # Normalize AOV by total orders
    user_stats['avg_order_value'] = user_stats['avg_order_value'] / user_stats['total_orders']
    
    # Heuristic for strict vegetarianism
    user_stats['is_veg_user'] = (user_stats['veg_item_ratio'] > 0.95).astype(int)
    
    return user_stats

user_features = extract_user_features(orders_df, items_df)
user_features.head()

## 3. Generating Contextual Cart Features
This is the most critical step. We must simulate the state of a cart *at the moment of prediction*. For the training set, we hold out the last item added to the cart as the positive label.

In [ ]:
def extract_cart_features(df):
    # Sort items sequentially within each order (simulating chronological adding to cart)
    df = df.sort_values(['order_id', 'item_id'])
    
    # Define chronological step within cart
    df['add_sequence'] = df.groupby('order_id').cumcount()
    
    # The Target Item: The last item added to the cart is our positive label
    target_items = df.groupby('order_id').last().reset_index()[['order_id', 'item_id']]
    target_items.rename(columns={'item_id': 'target_item_id'}, inplace=True)
    
    # The Context: Everything added to the cart BEFORE the target item
    context_df = df[df.groupby('order_id').cumcount(ascending=False) > 0].copy()
    
    cart_features = context_df.groupby('order_id').agg(
        current_cart_size=('item_id', 'count'),
        cart_total_value=('price', 'sum'),
        cart_veg_ratio=('is_veg', 'mean'),
        # In practice, we'd also track array of item categories here
    ).reset_index()
    
    return cart_features, target_items

merged_orders = orders_df.merge(items_df, on='item_id')
cart_features, true_labels = extract_cart_features(merged_orders)

print("Cart Features Distribution:")
cart_features.describe()

## 4. Negative Sampling
Our model learns to rank. For every positive item the user added, we must sample items they did *not* add as negative examples, allowing LightGBM to learn the contrast.

In [ ]:
def create_negative_samples(true_labels, all_item_ids, num_negatives=4):
    negatives = []
    for _, row in true_labels.iterrows():
        order_id = row['order_id']
        pos_item = row['target_item_id']
        
        # Randomly sample items not in the cart as negatives
        neg_items = np.random.choice([id for id in all_item_ids if id != pos_item], 
                                     size=num_negatives, replace=False)
        for neg in neg_items:
            negatives.append({'order_id': order_id, 'candidate_item_id': neg, 'label': 0})
            
    return pd.DataFrame(negatives)

# Note: In the actual preprocessing script, this is optimized via vectorized numpy operations
print("Negative sampling logic implemented.")

## Summary
We now have a methodology to extract user historical features, calculate real-time cart states, and perform negative sampling. These matrices are merged and saved to `data/processed/train_features.parquet` by the `preprocess.py` script for modeling.